# Modelo de Propensión a Hurto de Energía — Working Paper

**Miguel Ortiz C.** · Working paper · Julio 2026

---

Notebook autocontenido que reproduce el pipeline del modelo de propensión entrenado originalmente en Enel Distribución Chile y modernizado con datos sintéticos reproducibles. Entrena **4 modelos LightGBM**, uno por cluster de historial del cliente, y exporta las métricas, importancias y scores para el paper y el dashboard.

**Estructura:**
1. Setup y carga de datos
2. Feature engineering (target encoding, joins a nivel SED/SET/marca)
3. Segmentación por cluster y split temporal (train / test / deployment)
4. Entrenamiento de 4 modelos LightGBM con `scale_pos_weight`
5. Evaluación: AUC por cluster, precision@K, lift
6. Feature importance por cluster
7. Export de artefactos (métricas JSON, scores CSV, importancias)

**Requisitos:** `pandas`, `numpy`, `scikit-learn`, `lightgbm`, `matplotlib`, `seaborn`.

**Tiempo de ejecución:** ~2-4 minutos con los datos sintéticos.

## 1. Setup

In [1]:
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

import lightgbm as lgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

np.random.seed(42)

# Paths: busca la raiz del proyecto subiendo desde el cwd hasta encontrar
# el directorio que contiene data/synthetic. Asi el notebook funciona sin
# importar desde donde se ejecute, siempre que sea dentro del proyecto.
def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "data" / "synthetic").exists():
            return p
    return start
PROJECT_ROOT = _find_project_root(Path.cwd())
SYNTH_DIR = PROJECT_ROOT / "data" / "synthetic"
MAESTROS_DIR = SYNTH_DIR / "maestros"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = PROJECT_ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Paleta del paper
COL = {
    "ink": "#081630",
    "teal": "#3B878C",
    "teal_deep": "#125358",
    "teal_soft": "#D9E7E8",
    "grey": "#C2C3C5",
    "paper": "#EBEBED",
    "loss": "#A04545",
    "loss_soft": "#F0DCDA",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 150,
})

# Split temporal: train <= 202010, test 202011, deployment > 202011 (mismo criterio que el original)
SPLIT_TRAIN_END = 202010
SPLIT_TEST = 202011

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Setup OK · {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

PROJECT_ROOT: C:\Workspace\Optimizacion_de_Carteras\modelo_propension
Setup OK · 2026-07-29 21:45 UTC


## 2. Carga de datos y cruces

Se cargan las tablas (training + variables a nivel de red + historial de inspección) y se hacen los joins que el pipeline original hacía en SQL Server. Toda la lógica de merge se conserva.

In [2]:
print("Cargando tablas sinteticas ...")
df = pd.read_csv(SYNTH_DIR / "training_dataset.csv")
insp_hist = pd.read_csv(SYNTH_DIR / "variables_insp_hist.csv")
data_sed = pd.read_csv(SYNTH_DIR / "variables_nivel_sed.csv")
data_set = pd.read_csv(SYNTH_DIR / "variables_nivel_set.csv")
data_sa = pd.read_csv(SYNTH_DIR / "variables_nivel_set_alimentador.csv")
data_llave = pd.read_csv(SYNTH_DIR / "variables_nivel_llave.csv")
data_marca = pd.read_csv(SYNTH_DIR / "variables_nivel_marca.csv")
data_distrito = pd.read_csv(SYNTH_DIR / "variables_nivel_distrito.csv")
vars_con = pd.read_csv(SYNTH_DIR / "variables_consumo_con_notif.csv")
vars_sin = pd.read_csv(SYNTH_DIR / "variables_consumo_sin_notif.csv")
df_llave_sed = pd.read_csv(SYNTH_DIR / "llaves_por_sed.csv")

# Maestros para encoders
maestro_set = pd.read_csv(MAESTROS_DIR / "maestro_set.csv")
maestro_alimentador = pd.read_csv(MAESTROS_DIR / "maestro_alimentador.csv")
maestro_sed = pd.read_csv(MAESTROS_DIR / "maestro_sed.csv")
maestro_llave = pd.read_csv(MAESTROS_DIR / "maestro_llave.csv")
maestro_marca = pd.read_csv(MAESTROS_DIR / "maestro_marca.csv")
maestro_ali_sector_zona = pd.read_csv(MAESTROS_DIR / "maestro_ali_sector_zona.csv")

# --- Calcular variables derivadas en cada tabla de nivel (rates) ---
def add_rates(d, sufijo):
    """Calcula tasas / ratios por nivel a partir de los conteos."""
    d = d.copy()
    nro_insp_col = f"nro_inspecciones" if "nro_inspecciones" in d.columns else None
    nro_notif_col = f"nro_notificaciones" if "nro_notificaciones" in d.columns else None
    if nro_insp_col and nro_notif_col:
        d[f"efectividad_{sufijo}"] = d[nro_notif_col] / d[nro_insp_col].replace(0, np.nan)
    if "recupero" in d.columns and nro_notif_col:
        d[f"recup_prom_{sufijo}"] = d["recupero"] / d[nro_notif_col].replace(0, np.nan)
    if "nro_notif_causal4" in d.columns and nro_notif_col:
        d[f"perc_notif_causal4_{sufijo}"] = d["nro_notif_causal4"] / d[nro_notif_col].replace(0, np.nan)
    if "recupero_causal4" in d.columns and "nro_notif_causal4_val" in d.columns:
        d[f"recup_prom_causal4_{sufijo}"] = d["recupero_causal4"] / d["nro_notif_causal4_val"].replace(0, np.nan)
    if "nro_inspecciones" in d.columns and "nro_cuentas_insp" in d.columns:
        d[f"insp_por_cuenta_{sufijo}"] = d["nro_inspecciones"] / d["nro_cuentas_insp"].replace(0, np.nan)
    return d

data_sed = add_rates(data_sed, "sed")
data_set = add_rates(data_set, "set")
data_sa = add_rates(data_sa, "set_alimentador")
data_llave = add_rates(data_llave, "llave")
data_marca = add_rates(data_marca, "marca")
data_distrito = add_rates(data_distrito, "distrito")

# Tambien: vars de consumo — crear min_diff y cv_cons_*
for v in [vars_con, vars_sin]:
    if "q01_promedio_ant" in v.columns:
        v["dif_q01"] = v["q01_promedio_post"] - v["q01_promedio_ant"]
        v["dif_q02"] = v["q02_promedio_post"] - v["q02_promedio_ant"]
        v["dif_q03"] = v["q03_promedio_post"] - v["q03_promedio_ant"]
        v["min_diff"] = v[["dif_q01", "dif_q02", "dif_q03"]].min(axis=1)
    if "pro_cons_6m" in v.columns and "std_cons_6m" in v.columns:
        v["cv_cons_6m"] = v["std_cons_6m"] / v["pro_cons_6m"].replace(0, np.nan)
    if "pro_cons_12m" in v.columns and "std_cons_12m" in v.columns:
        v["cv_cons_12m"] = v["std_cons_12m"] / v["pro_cons_12m"].replace(0, np.nan)
    if "pro_cons_24m" in v.columns and "std_cons_24m" in v.columns:
        v["cv_cons_24m"] = v["std_cons_24m"] / v["pro_cons_24m"].replace(0, np.nan)
    if "pro_cons_6m" in v.columns and "pro_cons_12m" in v.columns:
        v["diff_cons_6m_12m"] = v["pro_cons_6m"] - v["pro_cons_12m"]
    if "pro_cons_6m" in v.columns and "pro_cons_24m" in v.columns:
        v["diff_cons_6m_24m"] = v["pro_cons_6m"] - v["pro_cons_24m"]
    if "pro_cons_12m" in v.columns and "pro_cons_24m" in v.columns:
        v["diff_cons_12m_24m"] = v["pro_cons_12m"] - v["pro_cons_24m"]

print(f"\ntraining_dataset: {df.shape}")
print(f"Clusters: {df['cluster'].value_counts().to_dict()}")

Cargando tablas sinteticas ...



training_dataset: (270000, 75)
Clusters: {'04.- SinInspecciones': 186173, '03.- ConInspeccionesSinIrregularidad': 54259, '02.- ConIrregularidadNoHurto': 16123, '01.- ConHurtoPrevio': 13445}


In [3]:
print("Aplicando cruces (joins) ...")

# El training_dataset sintetico ya viene con id_* (set, alimentador, sed, llave, marca, distrito, ali_sector_zona).
# Verificamos que existan y seguimos.
id_cols_required = ["id_set", "id_alimentador", "id_sed", "id_llave", "id_marca", "id_distrito", "id_ali_sector_zona"]
missing = [c for c in id_cols_required if c not in df.columns]
if missing:
    # Si falta alguno, hacemos el merge con el maestro correspondiente
    for cat, col_merge in [("set", "id_set"), ("alimentador", "id_alimentador"),
                           ("sed", "id_sed"), ("llave", "id_llave"), ("marca", "id_marca")]:
        if col_merge in missing:
            m = pd.read_csv(MAESTROS_DIR / f"maestro_{cat}.csv")
            df = df.drop(columns=[col_merge], errors="ignore").merge(m, on=cat, how="left")
    if "id_distrito" in missing:
        m = pd.read_csv(MAESTROS_DIR / "maestro_distrito.csv")
        df = df.drop(columns=["id_distrito"], errors="ignore").merge(
            m.rename(columns={"distrito": "cod_distrito", "id_distrito": "id_distrito"}),
            on="cod_distrito", how="left")
    if "id_ali_sector_zona" in missing:
        df = df.drop(columns=["id_ali_sector_zona"], errors="ignore").merge(
            maestro_ali_sector_zona[["alimentador", "sector", "zona", "id_ali_sector_zona"]],
            on=["alimentador", "sector", "zona"], how="left")

# Joins con tablas de nivel (join keys explicitos, no cartesiano).
# Solo se traen las variables del nivel; las join keys (periodo + nivel) ya estan en df.
JOINS = [
    (data_sed, "sed", ["periodo", "sed"]),
    (data_set, "set", ["periodo", "set"]),
    (data_sa, "set_alimentador", ["periodo", "set", "alimentador"]),
    (data_llave, "llave", ["periodo", "llave"]),
    (data_marca, "marca", ["periodo", "marca"]),
    (data_distrito, "distrito", ["periodo", "cod_distrito"]),
]
for tabla, sufijo, join_on in JOINS:
    var_cols = [c for c in tabla.columns if sufijo in c.lower() and c not in join_on]
    if var_cols:
        df = df.merge(tabla[join_on + var_cols], on=join_on, how="left")

# Join llaves_por_sed
df = df.merge(df_llave_sed, on=["periodo", "sed"], how="left")

print(f"\nDespues de cruces: {df.shape}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Columnas disponibles: {len(df.columns)}")
print(f"id_marca en df: {'id_marca' in df.columns}")

Aplicando cruces (joins) ...



Despues de cruces: (270000, 112)


Memoria: 472.0 MB
Columnas disponibles: 112
id_marca en df: True


In [4]:
# Variables derivadas simples (las que el codigo original calculaba con np.where y fechas)
df["fecha_instalacion"] = pd.to_datetime(df["fecha_instalacion"], errors="coerce")

# Ratio consumo / potencia (segun cluster — se hace en cada df_* mas abajo)
print("Variables derivadas listas.")

Variables derivadas listas.


## 3. Segmentación y split temporal

El pipeline original entrena **4 modelos LightGBM**, uno por cada cluster de historial del cliente. El split temporal sigue el criterio del trabajo original: train ≤ 202010, test = 202011, deployment > 202011.

In [5]:
CLUSTERS = [
    "01.- ConHurtoPrevio",
    "02.- ConIrregularidadNoHurto",
    "03.- ConInspeccionesSinIrregularidad",
    "04.- SinInspecciones",
]

# Variables finales por cluster (seleccion via feature importance + trend correlation en el original).
# Ajustadas al esquema del dataset sintetico (mismas semanticas, columnas que existen).
# IMPORTANTE: variables categoricas (string) reemplazadas por su id_* numerico para LightGBM.
COLS_01 = [
    "id_distrito", "id_marca",
    "notificaciones",
    "efectividad_marca", "recup_prom_marca",
    "consumos_dimi",
    "pro_cons_6m", "pro_cons_12m",
    "diff_cons_6m_12m",
    "casos_medidor_cambiado",
]
CAT_01 = ["id_distrito", "id_marca"]

COLS_02 = [
    "tipo_fase_M", "notif_causal1",
    "diff_cons_6m_12m",
    "efectividad_marca",
    "potencia",
]
CAT_02 = []

COLS_03 = [
    "tipo_fase_M", "potencia",
    "insp_efectivas", "insp_pendientes",
    "dist_ult_insp_efect",
    "casos_fact_U",
    "std_cons_6m",
    "ratio_cons_potencia_12m",
]
CAT_03 = []

COLS_04 = [
    "longitud",
    "sums_marca", "sums_distrito",
    "tipo_fase_M", "potencia",
    "casos_medidor_manip",
    "consumos_dimi",
    "dist_pri_lectura",
    "pro_cons_6m", "std_cons_6m",
    "min_diff",
    "diff_cons_6m_12m",
    "cv_cons_12m",
    "consumos_0",
]
CAT_04 = []

CLUSTERS_CONFIG = {
    "01.- ConHurtoPrevio": (COLS_01, CAT_01, vars_con),
    "02.- ConIrregularidadNoHurto": (COLS_02, CAT_02, vars_con),
    "03.- ConInspeccionesSinIrregularidad": (COLS_03, CAT_03, vars_sin),
    "04.- SinInspecciones": (COLS_04, CAT_04, vars_sin),
}

target = "target"
print(f"Clusters definidos: {list(CLUSTERS_CONFIG.keys())}")
print(f"\n# features por cluster:")
for cl, (cols, cat, _) in CLUSTERS_CONFIG.items():
    print(f"  {cl}: {len(cols)} ({len(cat)} cat)")

Clusters definidos: ['01.- ConHurtoPrevio', '02.- ConIrregularidadNoHurto', '03.- ConInspeccionesSinIrregularidad', '04.- SinInspecciones']

# features por cluster:
  01.- ConHurtoPrevio: 10 (2 cat)
  02.- ConIrregularidadNoHurto: 5 (0 cat)
  03.- ConInspeccionesSinIrregularidad: 8 (0 cat)
  04.- SinInspecciones: 14 (0 cat)


In [6]:
print("Construyendo dataframes por cluster con merge de variables de consumo ...")

dfs_cluster = {}
for cl, (cols_base, cat_feats, vars_cons) in CLUSTERS_CONFIG.items():
    # Clusters 03 y 04 requieren dist_pri_lectura y pro_cons_* (para ratios)
    extra_cols = []
    if cl in ("03.- ConInspeccionesSinIrregularidad", "04.- SinInspecciones"):
        if "dist_pri_lectura" in vars_cons.columns and "dist_pri_lectura" not in cols_base:
            extra_cols.append("dist_pri_lectura")
        for c in ["pro_cons_6m", "pro_cons_12m"]:
            if c in vars_cons.columns and c not in cols_base:
                extra_cols.append(c)

    # Todas las columnas de vars_cons necesarias (las del modelo + extras)
    needed_from_cons = [c for c in cols_base if c in vars_cons.columns and c not in df.columns]
    needed_from_cons += extra_cols
    needed_from_cons = list(set(needed_from_cons))

    # Subset: cols en df + merge con vars_cons
    cols_in_df = [c for c in cols_base if c in df.columns]
    sub = df.loc[df["cluster"] == cl,
                 ["periodo", "cuenta", "target"] + cols_in_df].copy()
    if needed_from_cons:
        sub = sub.merge(
            vars_cons[["periodo", "cuenta"] + needed_from_cons],
            on=["periodo", "cuenta"], how="left"
        )
    # Clusters 03 y 04 requieren dist_pri_lectura >= 6
    if cl in ("03.- ConInspeccionesSinIrregularidad", "04.- SinInspecciones"):
        sub = sub.loc[sub["dist_pri_lectura"] >= 6].copy()
    # Ratio consumo/potencia (variables derivadas que el original crea)
    if "pro_cons_6m" in sub.columns and "potencia" in sub.columns:
        sub["ratio_cons_potencia_6m"] = sub["pro_cons_6m"] / sub["potencia"].replace(0, np.nan)
    if "pro_cons_12m" in sub.columns and "potencia" in sub.columns:
        sub["ratio_cons_potencia_12m"] = sub["pro_cons_12m"] / sub["potencia"].replace(0, np.nan)
    dfs_cluster[cl] = sub
    print(f"  {cl}: {len(sub):>8,} filas · cols={sub.shape[1]} · target_rate={sub['target'].mean():.3f}")

Construyendo dataframes por cluster con merge de variables de consumo ...
  01.- ConHurtoPrevio:   13,445 filas · cols=13 · target_rate=0.558


  02.- ConIrregularidadNoHurto:   16,123 filas · cols=8 · target_rate=0.179


  03.- ConInspeccionesSinIrregularidad:   54,259 filas · cols=15 · target_rate=0.082


  04.- SinInspecciones:  186,173 filas · cols=20 · target_rate=0.038


## 4. Entrenamiento: 4 modelos LightGBM

Por cada cluster se entrena un `LGBMClassifier` con:
- `scale_pos_weight = N_neg / N_pos` para tratar el desbalance de clases
- Regularización explícita (`reg_alpha`, `reg_lambda`)
- `categorical_feature` para las variables categóricas de baja cardinalidad (cluster 01)
- Split train/test 80/20 estratificado, más split temporal para deployment (out-of-time)

In [7]:
# Hiperparametros por cluster (mismos que el original)
PARAMS = {
    "01.- ConHurtoPrevio": dict(n_estimators=300, learning_rate=0.075, subsample=0.80,
                                 colsample_bytree=0.90, reg_alpha=15.0, reg_lambda=20.0,
                                 max_depth=3, min_child_samples=50, num_leaves=30),
    "02.- ConIrregularidadNoHurto": dict(n_estimators=300, learning_rate=0.010, subsample=0.85,
                                          colsample_bytree=0.90, reg_alpha=9.0, reg_lambda=5.0,
                                          max_depth=3, min_child_samples=50, num_leaves=30),
    "03.- ConInspeccionesSinIrregularidad": dict(n_estimators=400, learning_rate=0.010, subsample=0.85,
                                                 colsample_bytree=0.90, reg_alpha=4.0, reg_lambda=5.0,
                                                 max_depth=3, min_child_samples=50, num_leaves=30),
    "04.- SinInspecciones": dict(n_estimators=400, learning_rate=0.090, subsample=0.85,
                                 colsample_bytree=0.90, reg_alpha=4.0, reg_lambda=5.0,
                                 max_depth=3, min_child_samples=50, num_leaves=30),
}

print("Hiperparametros definidos por cluster.")

Hiperparametros definidos por cluster.


In [8]:
results = {}
feature_importances = {}
t_start_total = datetime.now()

for cl in CLUSTERS:
    print(f"\n--- {cl} ---")
    cols, cat_feats, _ = CLUSTERS_CONFIG[cl]
    sub = dfs_cluster[cl]
    params = PARAMS[cl]

    if len(sub) == 0:
        print(f"  AVISO: sub vacio, saltando {cl}")
        continue

    # Filtrar a columnas que existen en sub
    cols_avail = [c for c in cols if c in sub.columns]
    if len(cols_avail) < len(cols):
        print(f"  AVISO: solo {len(cols_avail)}/{len(cols)} cols disponibles en sub")

    # Split temporal
    train_mask = sub["periodo"] <= SPLIT_TRAIN_END
    test_mask = sub["periodo"] == SPLIT_TEST
    dep_mask = sub["periodo"] > SPLIT_TEST

    # Train set: periodo <= 202010, dropna SOLO sobre las cols_avail
    train_data = sub.loc[train_mask, cols_avail + [target]].dropna()
    print(f"  train pool: {len(train_data):,} filas (de {train_mask.sum():,} con periodo <= {SPLIT_TRAIN_END})")
    if len(train_data) < 50 or train_data[target].nunique() < 2:
        print(f"  AVISO: train_data muy pequeno o sin varianza, saltando {cl}")
        continue

    X = train_data[cols_avail]; y = train_data[target]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=0)

    spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    print(f"  train: {len(X_tr):>7,} · test: {len(X_te):>6,} · scale_pos_weight={spw:.1f}")

    cat_use = [c for c in cat_feats if c in cols_avail]
    clf = lgb.LGBMClassifier(importance_type="gain", scale_pos_weight=spw, **params, verbose=-1)
    clf.fit(X_tr, y_tr, categorical_feature=cat_use if cat_use else "auto")

    # AUC train / test
    y_pred_tr = clf.predict_proba(X_tr)[:, 1]
    y_pred_te = clf.predict_proba(X_te)[:, 1]
    auc_tr = roc_auc_score(y_tr, y_pred_tr)
    auc_te = roc_auc_score(y_te, y_pred_te)

    # AUC out-of-time
    dep_data = sub.loc[dep_mask, cols_avail + [target]].dropna()
    if len(dep_data) > 0 and dep_data[target].nunique() > 1:
        X_dep, y_dep = dep_data[cols_avail], dep_data[target]
        y_pred_dep = clf.predict_proba(X_dep)[:, 1]
        auc_dep = roc_auc_score(y_dep, y_pred_dep)
    else:
        auc_dep = float("nan")

    print(f"  AUC train={auc_tr:.3f} · test={auc_te:.3f} · OOT deploy={auc_dep:.3f}")

    fi = pd.DataFrame({
        "feature": cols_avail,
        "importance": clf.feature_importances_,
    }).sort_values("importance", ascending=False)
    feature_importances[cl] = fi

    results[cl] = {
        "n_train": int(len(X_tr)),
        "n_test": int(len(X_te)),
        "n_deploy": int(len(dep_data)),
        "scale_pos_weight": spw,
        "auc_train": float(auc_tr),
        "auc_test": float(auc_te),
        "auc_deploy": float(auc_dep),
        "y_pred_test": y_pred_te.tolist()[:200],
        "y_test_sample": y_te.tolist()[:200],
        "n_features": len(cols_avail),
    }

print(f"\nEntrenamiento completo en {(datetime.now() - t_start_total).total_seconds():.1f}s")


--- 01.- ConHurtoPrevio ---
  train pool: 12,328 filas (de 12,328 con periodo <= 202010)
  train:   9,862 · test:  2,466 · scale_pos_weight=0.8


  AUC train=0.594 · test=0.487 · OOT deploy=0.489

--- 02.- ConIrregularidadNoHurto ---
  train pool: 14,779 filas (de 14,779 con periodo <= 202010)
  train:  11,823 · test:  2,956 · scale_pos_weight=4.6
  AUC train=0.500 · test=0.500 · OOT deploy=0.500

--- 03.- ConInspeccionesSinIrregularidad ---
  train pool: 49,726 filas (de 49,726 con periodo <= 202010)
  train:  39,780 · test:  9,946 · scale_pos_weight=11.1


  AUC train=0.586 · test=0.499 · OOT deploy=0.503

--- 04.- SinInspecciones ---
  train pool: 170,795 filas (de 170,795 con periodo <= 202010)
  train: 136,636 · test: 34,159 · scale_pos_weight=25.3


  AUC train=0.725 · test=0.503 · OOT deploy=0.489

Entrenamiento completo en 5.0s


## 5. Resultados por cluster

In [9]:
print("=" * 70)
print("RESULTADOS POR CLUSTER")
print("=" * 70)
print(f"\n{'Cluster':<42s} {'N_train':>9s} {'N_test':>7s} {'AUC_tr':>7s} {'AUC_te':>7s} {'AUC_OOT':>7s}")
print("-" * 90)
for cl, r in results.items():
    print(f"{cl:<42s} {r['n_train']:>9,d} {r['n_test']:>7,d} "
          f"{r['auc_train']:>7.3f} {r['auc_test']:>7.3f} {r['auc_deploy']:>7.3f}")

RESULTADOS POR CLUSTER

Cluster                                      N_train  N_test  AUC_tr  AUC_te AUC_OOT
------------------------------------------------------------------------------------------
01.- ConHurtoPrevio                            9,862   2,466   0.594   0.487   0.489
02.- ConIrregularidadNoHurto                  11,823   2,956   0.500   0.500   0.500
03.- ConInspeccionesSinIrregularidad          39,780   9,946   0.586   0.499   0.503
04.- SinInspecciones                         136,636  34,159   0.725   0.503   0.489


In [10]:
# Precision@K y lift por cluster (top decil)
print("\nPrecision@K y lift operacional (top 10% de scores):")
print(f"\n{'Cluster':<42s} {'Base rate':>10s} {'P@10%':>8s} {'Lift':>7s}")
print("-" * 70)

precision_at_k = {}
for cl in CLUSTERS:
    cols, cat_feats, _ = CLUSTERS_CONFIG[cl]
    sub = dfs_cluster[cl]
    if len(sub) == 0:
        continue
    cols_avail = [c for c in cols if c in sub.columns]
    train_mask = sub["periodo"] <= SPLIT_TRAIN_END
    dep_mask = sub["periodo"] > SPLIT_TEST

    # Reentrenar rapido para predecir en deploy
    train_data = sub.loc[train_mask, cols_avail + [target]].dropna()
    if len(train_data) < 50 or train_data[target].nunique() < 2:
        continue
    X = train_data[cols_avail]; y = train_data[target]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=0)
    spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    cat_use = [c for c in cat_feats if c in cols_avail]
    clf = lgb.LGBMClassifier(importance_type="gain", scale_pos_weight=spw, **PARAMS[cl], verbose=-1)
    clf.fit(X_tr, y_tr, categorical_feature=cat_use if cat_use else "auto")

    dep_data = sub.loc[dep_mask, cols_avail + [target]].dropna()
    if len(dep_data) == 0 or dep_data[target].nunique() < 2:
        continue
    y_pred_dep = clf.predict_proba(dep_data[cols_avail])[:, 1]
    y_dep = dep_data[target].values

    base_rate = float(y_dep.mean())
    n_top = max(int(len(y_dep) * 0.10), 1)
    top_idx = np.argsort(y_pred_dep)[-n_top:]
    precision_topk = float(y_dep[top_idx].mean())
    lift = precision_topk / base_rate if base_rate > 0 else float("nan")

    precision_at_k[cl] = {
        "base_rate": base_rate, "precision_at_10pct": precision_topk, "lift": lift,
        "n_deploy": int(len(y_dep)),
    }
    print(f"{cl:<42s} {base_rate:>10.3f} {precision_topk:>8.3f} {lift:>7.2f}x")


Precision@K y lift operacional (top 10% de scores):

Cluster                                     Base rate    P@10%    Lift
----------------------------------------------------------------------
01.- ConHurtoPrevio                             0.554    0.611    1.10x
02.- ConIrregularidadNoHurto                    0.177    0.224    1.26x


03.- ConInspeccionesSinIrregularidad            0.079    0.092    1.16x


04.- SinInspecciones                            0.038    0.042    1.10x


## 6. Feature importance por cluster

In [11]:
print("Top 10 features por cluster (importance = gain acumulado):")
print()
for cl, fi in feature_importances.items():
    print(f"\n  {cl}:")
    for _, row in fi.head(10).iterrows():
        bar = "#" * int(row["importance"] / max(fi["importance"].max(), 1) * 30)
        print(f"    {row['feature']:<35s} {row['importance']:>6.0f}  {bar}")

Top 10 features por cluster (importance = gain acumulado):


  01.- ConHurtoPrevio:
    id_distrito                            111  ##############################
    pro_cons_12m                            62  ################
    diff_cons_6m_12m                        58  ###############
    id_marca                                51  #############
    efectividad_marca                       48  ############
    recup_prom_marca                        48  ############
    pro_cons_6m                             40  ##########
    casos_medidor_cambiado                   8  ##
    notificaciones                           4  #
    consumos_dimi                            2  

  02.- ConIrregularidadNoHurto:
    tipo_fase_M                              0  
    notif_causal1                            0  
    diff_cons_6m_12m                         0  
    efectividad_marca                        0  
    potencia                                 0  

  03.- ConInspeccionesSinIrregularid

In [12]:
# F1 — Composicion de la base por cluster
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

n_por_cluster = {cl: len(dfs_cluster[cl]) for cl in CLUSTERS}
rates = {cl: float(dfs_cluster[cl]["target"].mean()) for cl in CLUSTERS}
colors_cl = [COL["loss"], COL["teal"], COL["teal_deep"], COL["grey"]]
short_labels = ["ConHurto\nPrevio", "ConIrreg.\nNoHurto", "ConInsp.\nSinIrreg.", "SinInspec."]

ax = axes[0]
vals = [n_por_cluster[c] for c in CLUSTERS]
ax.bar(short_labels, vals, color=colors_cl, edgecolor="white", linewidth=1.2)
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9, color=COL["ink"], weight="bold")
ax.set_title("Tamano de la base por cluster")
ax.set_ylabel("Cantidad de cuentas-periodo")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y/1000):,}k"))

ax = axes[1]
vals = [rates[c] for c in CLUSTERS]
bars = ax.bar(short_labels, [v*100 for v in vals], color=colors_cl, edgecolor="white", linewidth=1.2)
for i, v in enumerate(vals):
    ax.text(i, v*100, f"{v*100:.1f}%", ha="center", va="bottom", fontsize=9, color=COL["ink"], weight="bold")
ax.set_title("Tasa de hurto real (target) por cluster")
ax.set_ylabel("Target rate (%)")
ax.axhline(np.mean(list(rates.values()))*100, color=COL["grey"], linestyle="--", linewidth=1, alpha=0.7)
ax.text(0.05, 0.92, f"Promedio: {np.mean(list(rates.values()))*100:.1f}%", transform=ax.transAxes, fontsize=8, color=COL["grey"])

fig.suptitle("Composicion del universo de entrenamiento", fontsize=13, weight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_segmentos.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig1_segmentos.png")

OK fig1_segmentos.png


In [13]:
# F2 — Feature importance por cluster (top 10 cada uno, panel 2x2)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for idx, cl in enumerate(CLUSTERS):
    ax = axes[idx // 2, idx % 2]
    if cl not in feature_importances:
        ax.text(0.5, 0.5, f"(sin datos)\n{cl}", ha="center", va="center", transform=ax.transAxes, fontsize=10, color=COL["grey"])
        ax.axis("off")
        continue
    fi = feature_importances[cl].head(10).sort_values("importance")
    ax.barh(fi["feature"], fi["importance"], color=colors_cl[idx], edgecolor="white")
    ax.set_title(cl, fontsize=10, loc="left")
    ax.set_xlabel("Importancia (gain)")
    ax.grid(axis="x", alpha=0.3)

fig.suptitle("Top 10 features por modelo (importance = gain)", fontsize=13, weight="bold", y=1.00)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig2_feature_importance.png")

OK fig2_feature_importance.png


In [14]:
# F3 — AUC comparativo (train / test / out-of-time) por cluster
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLUSTERS))
width = 0.27
auc_tr = [results[c]["auc_train"] for c in CLUSTERS]
auc_te = [results[c]["auc_test"] for c in CLUSTERS]
auc_dep = [results[c]["auc_deploy"] for c in CLUSTERS]
ax.bar(x - width, auc_tr, width, label="Train", color=COL["teal"], edgecolor="white")
ax.bar(x, auc_te, width, label="Test (in-sample 202011)", color=COL["teal_deep"], edgecolor="white")
ax.bar(x + width, auc_dep, width, label="Deployment (out-of-time)", color=COL["loss"], edgecolor="white")

ax.axhline(0.5, color=COL["grey"], linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=9)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("AUC-ROC")
ax.set_title("AUC por cluster y horizonte de evaluacion")
ax.legend(loc="lower left", fontsize=9)

for i, c in enumerate(CLUSTERS):
    ax.text(i - width, auc_tr[i] + 0.01, f"{auc_tr[i]:.2f}", ha="center", fontsize=8, color=COL["ink"])
    ax.text(i, auc_te[i] + 0.01, f"{auc_te[i]:.2f}", ha="center", fontsize=8, color=COL["ink"])
    ax.text(i + width, auc_dep[i] + 0.01, f"{auc_dep[i]:.2f}", ha="center", fontsize=8, color=COL["ink"])
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_auc_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig3_auc_comparison.png")

OK fig3_auc_comparison.png


In [15]:
# F4 — Distribucion de scores y precision@K (lift operacional)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
labels_cl = short_labels

# Helper para predecir en deploy
def predict_deploy(cl):
    cols, cat_feats, _ = CLUSTERS_CONFIG[cl]
    sub = dfs_cluster[cl]
    if len(sub) == 0: return None, None, None
    cols_avail = [c for c in cols if c in sub.columns]
    train_mask = sub["periodo"] <= SPLIT_TRAIN_END
    dep_mask = sub["periodo"] > SPLIT_TEST
    train_data = sub.loc[train_mask, cols_avail + [target]].dropna()
    if len(train_data) < 50 or train_data[target].nunique() < 2: return None, None, None
    X = train_data[cols_avail]; y = train_data[target]
    X_tr, _, y_tr, _ = train_test_split(X, y, test_size=0.20, stratify=y, random_state=0)
    spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    cat_use = [c for c in cat_feats if c in cols_avail]
    clf = lgb.LGBMClassifier(importance_type="gain", scale_pos_weight=spw, **PARAMS[cl], verbose=-1)
    clf.fit(X_tr, y_tr, categorical_feature=cat_use if cat_use else "auto")
    dep_data = sub.loc[dep_mask, cols_avail + [target]].dropna()
    if len(dep_data) == 0 or dep_data[target].nunique() < 2: return None, None, None
    y_pred_dep = clf.predict_proba(dep_data[cols_avail])[:, 1]
    return y_pred_dep, dep_data[target].values, clf

# Re-generar scores en deploy para ver distribucion
for idx, cl in enumerate(CLUSTERS):
    y_pred_dep, y_dep, _ = predict_deploy(cl)
    if y_pred_dep is None: continue
    ax.hist(y_pred_dep, bins=40, alpha=0.55, color=colors_cl[idx], label=labels_cl[idx], density=True)
ax.set_xlabel("Score de propension")
ax.set_ylabel("Densidad")
ax.set_title("Distribucion de scores (deployment)")
ax.legend(loc="upper right", fontsize=8)

ax = axes[1]
labels_p = short_labels
base_rates = [precision_at_k[c]["base_rate"]*100 for c in CLUSTERS if c in precision_at_k]
prec_at_k = [precision_at_k[c]["precision_at_10pct"]*100 for c in CLUSTERS if c in precision_at_k]
n_ok = len(base_rates)
x = np.arange(n_ok)
width = 0.35
ax.bar(x - width/2, base_rates, width, label="Base rate (azar)", color=COL["grey"], edgecolor="white")
ax.bar(x + width/2, prec_at_k, width, label="Precision@10% (modelo)", color=COL["teal"], edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([labels_p[i] for i, c in enumerate(CLUSTERS) if c in precision_at_k], fontsize=9)
ax.set_ylabel("Tasa de hurto real (%)")
ax.set_title("Precision operacional: top 10% de scores vs azar")
ax.legend(loc="upper right", fontsize=9)
for i, (br, pk) in enumerate(zip(base_rates, prec_at_k)):
    ax.text(i - width/2, br + 0.3, f"{br:.1f}%", ha="center", fontsize=8)
    ax.text(i + width/2, pk + 0.3, f"{pk:.1f}%", ha="center", fontsize=8, weight="bold")
    # Get lift for this cluster
    cluster_idx = [j for j, c in enumerate(CLUSTERS) if c in precision_at_k][i]
    cl_here = [c for c in CLUSTERS if c in precision_at_k][i]
    lift = precision_at_k[cl_here]["lift"]
    ax.text(i, max(br, pk) + 4, f"lift {lift:.1f}x", ha="center", fontsize=8, color=COL["loss"], style="italic")

fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_scores_y_precision.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig4_scores_y_precision.png")

OK fig4_scores_y_precision.png


In [16]:
# F5 — Heatmap de grupos de variables y su contribution promedio
GROUPS = {
    "Geografia": ["longitud", "latitud", "distancia_sed", "distancia_set", "distancia_sed_set"],
    "Categoria red": ["id_set", "id_alimentador", "id_sed", "id_llave", "id_marca",
                       "id_ali_sector_zona", "cod_distrito"],
    "Historial cliente": ["tipo_fase_M", "tipo_acomet_S", "potencia", "tiempo_ult_cambio_med"],
    "Inspeccion": ["insp_efectivas", "insp_pendientes", "notificaciones", "notif_causal1",
                   "notif_causal4", "casos_medidor_cambiado", "casos_medidor_manip"],
    "Contexto red": ["efectividad_marca", "insp_por_cuenta_marca", "nro_notif_causal4_sed",
                     "efectividad_distrito", "perc_recup_causal4_marca", "recup_prom_causal4_marca"],
    "Consumo": ["pro_cons_6m", "pro_cons_12m", "std_cons_6m", "ratio_cons_potencia_12m",
                "diff_cons_6m_12m", "min_diff", "consumos_0", "consumos_dimi"],
}

matrix = np.zeros((len(CLUSTERS), len(GROUPS)))
for i, cl in enumerate(CLUSTERS):
    fi = feature_importances[cl].set_index("feature")["importance"]
    total = fi.sum()
    for j, (group_name, feats) in enumerate(GROUPS.items()):
        s = sum(fi.get(f, 0) for f in feats)
        matrix[i, j] = (s / total * 100) if total > 0 else 0

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.Blues
im = ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=0, vmax=max(8, matrix.max()))
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels(list(GROUPS.keys()), rotation=20, ha="right", fontsize=10)
ax.set_yticks(range(len(CLUSTERS)))
ax.set_yticklabels(short_labels, fontsize=10)
for i in range(len(CLUSTERS)):
    for j in range(len(GROUPS)):
        v = matrix[i, j]
        ax.text(j, i, f"{v:.1f}%", ha="center", va="center",
                color="white" if v > matrix.max() * 0.5 else COL["ink"],
                fontsize=9, weight="bold")
ax.set_title("Contribucion por grupo de variables al modelo (% del gain total)", pad=14)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label="% del gain")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_grupos_variables.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig5_grupos_variables.png")

OK fig5_grupos_variables.png


In [17]:
# F6 — Curvas ROC por cluster (sobre deployment)
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))
for idx, cl in enumerate(CLUSTERS):
    y_pred_dep, y_dep, _ = predict_deploy(cl)
    if y_pred_dep is None: continue
    fpr, tpr, _ = roc_curve(y_dep, y_pred_dep)
    auc = roc_auc_score(y_dep, y_pred_dep)
    ax.plot(fpr, tpr, label=f"{short_labels[idx].replace(chr(10), ' ')} (AUC={auc:.3f})",
            color=colors_cl[idx], linewidth=2)
ax.plot([0, 1], [0, 1], "--", color=COL["grey"], linewidth=0.8, label="Azar")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC — evaluacion out-of-time (deployment)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig6_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig6_roc_curves.png")

OK fig6_roc_curves.png


In [18]:
# F7 — Estabilidad temporal del AUC (por periodo, out-of-time)
fig, ax = plt.subplots(figsize=(11, 4.5))
all_periods = sorted(df["periodo"].unique())
deploy_periods = [p for p in all_periods if p > SPLIT_TEST]
x_p = np.arange(len(deploy_periods))
for idx, cl in enumerate(CLUSTERS):
    cols, cat_feats, _ = CLUSTERS_CONFIG[cl]
    sub = dfs_cluster[cl]
    if len(sub) == 0: continue
    cols_avail = [c for c in cols if c in sub.columns]
    train_mask = sub["periodo"] <= SPLIT_TRAIN_END
    train_data = sub.loc[train_mask, cols_avail + [target]].dropna()
    if len(train_data) < 50 or train_data[target].nunique() < 2: continue
    X = train_data[cols_avail]; y = train_data[target]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=0)
    spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    cat_use = [c for c in cat_feats if c in cols_avail]
    clf = lgb.LGBMClassifier(importance_type="gain", scale_pos_weight=spw, **PARAMS[cl], verbose=-1)
    clf.fit(X_tr, y_tr, categorical_feature=cat_use if cat_use else "auto")

    aucs = []
    for p in deploy_periods:
        d = sub.loc[sub["periodo"] == p, cols_avail + [target]].dropna()
        if len(d) < 20 or d[target].nunique() < 2:
            aucs.append(np.nan); continue
        yp = clf.predict_proba(d[cols_avail])[:, 1]
        try:
            aucs.append(roc_auc_score(d[target], yp))
        except Exception:
            aucs.append(np.nan)
    ax.plot(x_p, aucs, marker="o", color=colors_cl[idx],
            label=short_labels[idx].replace("\n", " "), linewidth=2, markersize=5)

ax.axhline(0.5, color=COL["grey"], linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_xticks(x_p)
ax.set_xticklabels([str(p) for p in deploy_periods], rotation=30, fontsize=8, family="monospace")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("AUC-ROC (mes a mes)")
ax.set_title("Estabilidad temporal del AUC por cluster (out-of-time)")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig7_auc_temporal.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig7_auc_temporal.png")

OK fig7_auc_temporal.png


In [19]:
# F8 — Numero de features y N por cluster (tabla visual)
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")
table_data = []
for cl, (cols, cat_feats, _) in CLUSTERS_CONFIG.items():
    r = results[cl]
    pk = precision_at_k[cl]
    table_data.append([
        cl,
        f"{r['n_train']:,}",
        f"{r['n_test']:,}",
        f"{r['n_deploy']:,}",
        f"{r['scale_pos_weight']:.0f}",
        f"{len(cols)} ({len(cat_feats)} cat)",
        f"{r['auc_train']:.3f}",
        f"{r['auc_test']:.3f}",
        f"{r['auc_deploy']:.3f}",
        f"{pk['precision_at_10pct']*100:.1f}%",
        f"{pk['lift']:.1f}x",
    ])
headers = ["Cluster", "N train", "N test", "N deploy", "SPW", "# feats",
           "AUC tr", "AUC te", "AUC OOT", "P@10%", "Lift"]
table = ax.table(cellText=table_data, colLabels=headers, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)
# Header
for j in range(len(headers)):
    cell = table[(0, j)]
    cell.set_facecolor(COL["teal_deep"])
    cell.set_text_props(color="white", weight="bold", family="monospace")
# Color por cluster
for i in range(len(CLUSTERS)):
    for j in range(len(headers)):
        cell = table[(i + 1, j)]
        cell.set_facecolor([COL["loss_soft"], COL["teal_soft"], COL["paper"], COL["paper"]][i])
        cell.set_text_props(family="monospace")
    # Cluster label en bold
    table[(i + 1, 0)].set_text_props(weight="bold")
ax.set_title("Tabla resumen: metricas por cluster", pad=20, fontsize=13, weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig8_tabla_resumen.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("OK fig8_tabla_resumen.png")

OK fig8_tabla_resumen.png


## 7. Export de artefactos

Se exportan los resultados a JSON (para el dashboard HTML) y CSV (para análisis posterior).

In [20]:
# Export de metricas, importancias y precision@K
metrics_json = {
    "metadata": {
        "fecha": datetime.now(timezone.utc).isoformat(),
        "split_train_end": SPLIT_TRAIN_END,
        "split_test": SPLIT_TEST,
        "modelo": "LGBMClassifier",
        "version": "1.0.0-synthetic",
    },
    "results": {cl: {k: v for k, v in r.items() if k not in ("y_pred_test", "y_test_sample")} for cl, r in results.items()},
    "precision_at_k": precision_at_k,
    "feature_importance": {
        cl: fi.to_dict(orient="records") for cl, fi in feature_importances.items()
    },
    "hiperparametros": {cl: {k: v for k, v in p.items()} for cl, p in PARAMS.items()},
    "segmentos": {
        cl: {
            "n": int(len(dfs_cluster[cl])),
            "target_rate": float(dfs_cluster[cl]["target"].mean()),
        } for cl in CLUSTERS
    },
}
out_metrics = PROC_DIR / "metrics.json"
out_metrics.write_text(json.dumps(metrics_json, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"OK metrics.json ({out_metrics.stat().st_size:,} bytes)")

# Export de feature importance por cluster (CSV)
for cl, fi in feature_importances.items():
    safe = cl.split(".")[0]
    fi.to_csv(PROC_DIR / f"feature_importance_{safe}.csv", index=False)
print("OK feature_importance_*.csv")

# Export de segmentacion
seg_df = pd.DataFrame([
    {
        "cluster": cl,
        "label_short": short_labels[i],
        "n_total": int(len(dfs_cluster[cl])),
        "target_rate": float(dfs_cluster[cl]["target"].mean()),
        "n_features": len(CLUSTERS_CONFIG[cl][0]),
        "auc_train": results[cl]["auc_train"],
        "auc_test": results[cl]["auc_test"],
        "auc_deploy": results[cl]["auc_deploy"],
        "precision_at_10pct": precision_at_k[cl]["precision_at_10pct"],
        "lift": precision_at_k[cl]["lift"],
    }
    for i, cl in enumerate(CLUSTERS)
])
seg_df.to_csv(PROC_DIR / "segments_summary.csv", index=False)
print("OK segments_summary.csv")

# Resumen final
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"Total filas entrenamiento: {len(df):,}")
print(f"Clusters: {len(CLUSTERS)}")
print(f"AUC promedio (deployment): {np.mean([r['auc_deploy'] for r in results.values()]):.3f}")
print(f"Lift promedio top 10%: {np.mean([p['lift'] for p in precision_at_k.values()]):.2f}x")
print(f"Figuras generadas: {len(list(FIG_DIR.glob('*.png')))}")
print(f"Outputs en: {PROC_DIR}")

OK metrics.json (7,569 bytes)
OK feature_importance_*.csv
OK segments_summary.csv

RESUMEN FINAL
Total filas entrenamiento: 270,000
Clusters: 4
AUC promedio (deployment): 0.495
Lift promedio top 10%: 1.16x
Figuras generadas: 8
Outputs en: C:\Workspace\Optimizacion_de_Carteras\modelo_propension\data\processed
